# SurfaceTarget-CRC — phase 04 — patch 4 5

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.5
#
# FINAL HGNC-VALIDATED DEPMAP DEPENDENCY INTEGRATION
#
# Objectives:
# 1. Validate previously saved analysis inputs
# 2. Confirm DepMap source checksums
# 3. Import 13 newly resolved HGNC gene mappings
# 4. Extract the corresponding CRISPR measurements
# 5. Calculate corrected CRC dependency statistics
# 6. Calculate patient-deduplicated sensitivity statistics
# 7. Compare with the established other-lineage cohort
# 8. Preserve all 930 original dependency results
# 9. Retain 5 genes without DepMap measurements
# 10. Save final Phase 4 tables and provenance
#
# IMPORTANT:
# - No original results overwritten
# - No gene symbols guessed
# - No MSS status inferred
# - No automatic candidate exclusions
# - No therapeutic target ranking
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS AND CONFIGURATION
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

import hashlib
import json
import math


ROOT = Path("/content/SurfaceTarget-CRC")

DEPMAP = ROOT / "data/raw/depmap/26Q3"

META = ROOT / "data/metadata"

RESULTS = ROOT / "results/tables"

RESULTS.mkdir(parents=True, exist_ok=True)
META.mkdir(parents=True, exist_ok=True)


CRISPR_PATH = DEPMAP / "CRISPRGeneEffect.csv"

MODEL_PATH = DEPMAP / "Model.csv"

IMPORT_PATH = (
    META /
    "depmap_26q3_import_verification.csv"
)

CROSSWALK_PATH = (
    RESULTS /
    "depmap_26q3_hgnc_crosswalk_audit.csv"
)

PREVIOUS_SUMMARY_PATH = (
    RESULTS /
    "depmap_26q3_refined_dependency_summary.csv"
)

CRC_MODELS_PATH = (
    RESULTS /
    "depmap_26q3_refined_crc_models.csv"
)

DEDUP_MODELS_PATH = (
    RESULTS /
    "depmap_26q3_patient_deduplicated_models.csv"
)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.5")
print("FINAL DEPMAP DEPENDENCY INTEGRATION")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE REQUIRED FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING PHASE 4 RESEARCH INPUTS")
print("-" * 65)


required_paths = [
    CRISPR_PATH,
    MODEL_PATH,
    IMPORT_PATH,
    CROSSWALK_PATH,
    PREVIOUS_SUMMARY_PATH,
    CRC_MODELS_PATH,
    DEDUP_MODELS_PATH
]


for path in required_paths:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing: {path}"
        )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


import_audit = pd.read_csv(
    IMPORT_PATH,
    dtype=str,
    keep_default_na=False
)


if import_audit["filename"].duplicated().any():

    raise RuntimeError(
        "Duplicate source files in import audit."
    )


import_audit = import_audit.set_index(
    "filename"
)


for path in [MODEL_PATH, CRISPR_PATH]:

    if path.name not in import_audit.index:

        raise RuntimeError(
            f"Missing source checksum: {path.name}"
        )

    expected_hash = import_audit.loc[
        path.name,
        "sha256"
    ]

    actual_hash = sha256_file(path)

    if actual_hash != expected_hash:

        raise RuntimeError(
            f"Source checksum mismatch: {path.name}"
        )


print("\nLocally recorded source checksums: PASSED")

print(
    "Official DepMap source checksum "
    "comparison remains unavailable."
)


# ------------------------------------------------------------
# 3. LOAD HGNC-VALIDATED GENE MAPPINGS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING HGNC-VALIDATED CANDIDATE MAPPINGS")
print("-" * 65)


crosswalk = pd.read_csv(
    CROSSWALK_PATH,
    dtype=str,
    keep_default_na=False
)


required_columns = {
    "gene_id",
    "ensembl_stable_id",
    "gene_name",
    "previous_depmap_column",
    "hgnc_entrez_id",
    "proposed_depmap_column",
    "proposed_depmap_symbol",
    "identity_validation_status"
}


if not required_columns.issubset(
    crosswalk.columns
):

    raise RuntimeError(
        "Unexpected HGNC crosswalk schema."
    )


if len(crosswalk) != 948:

    raise RuntimeError(
        "Expected 948 research genes."
    )


if crosswalk["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate research gene IDs."
    )


status_counts = (
    crosswalk["identity_validation_status"]
    .value_counts()
)


print("\nIdentifier validation distribution:")

print(status_counts.to_string())


verified = crosswalk[
    crosswalk["identity_validation_status"]
    .eq("Verified existing mapping")
].copy()


newly_resolved = crosswalk[
    crosswalk["identity_validation_status"]
    .eq("Newly resolved by NCBI Gene ID")
].copy()


unavailable = crosswalk[
    crosswalk["identity_validation_status"]
    .eq("HGNC NCBI Gene ID absent from DepMap")
].copy()


if len(verified) != 930:

    raise RuntimeError(
        "Expected 930 verified original mappings."
    )


if len(newly_resolved) != 13:

    raise RuntimeError(
        "Expected 13 newly resolved mappings."
    )


if len(unavailable) != 5:

    raise RuntimeError(
        "Expected five genes absent by HGNC "
        "NCBI Gene ID from DepMap."
    )


if 930 + 13 + 5 != len(crosswalk):

    raise RuntimeError(
        "Unexpected identifier-validation categories."
    )


print("\nVerified existing genes:", len(verified))

print("Newly resolved genes:", len(newly_resolved))

print("Genes without DepMap columns:", len(unavailable))


# ------------------------------------------------------------
# 4. VERIFY COMPLETE GENE MAPPING UNIQUENESS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING FINAL GENE COLUMN CORRESPONDENCE")
print("-" * 65)


mapped = crosswalk[
    crosswalk["identity_validation_status"]
    .isin([
        "Verified existing mapping",
        "Newly resolved by NCBI Gene ID"
    ])
].copy()


if mapped[
    "proposed_depmap_column"
].eq("").any():

    raise RuntimeError(
        "Mapped genes contain empty DepMap columns."
    )


if mapped[
    "proposed_depmap_column"
].duplicated().any():

    raise RuntimeError(
        "Multiple research genes map to "
        "the same DepMap gene column."
    )


if mapped["hgnc_entrez_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate NCBI Gene IDs among mapped genes."
    )


if len(mapped) != 943:

    raise RuntimeError(
        "Expected 943 uniquely mapped genes."
    )


# Verify previously established mappings
# were not changed by HGNC integration.

if not (
    verified["previous_depmap_column"]
    == verified["proposed_depmap_column"]
).all():

    raise RuntimeError(
        "Previously verified gene columns changed."
    )


print("\nFinal verified DepMap genes:", len(mapped))

print("Existing mappings unchanged: PASSED")

print("Gene-column uniqueness: PASSED")


# ------------------------------------------------------------
# 5. VERIFY THE 13 NEW DEPMAP COLUMNS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING NEWLY RESOLVED CRISPR COLUMNS")
print("-" * 65)


crispr_header = pd.read_csv(
    CRISPR_PATH,
    nrows=0
).columns.tolist()


if len(crispr_header) != 18435:

    raise RuntimeError(
        "Unexpected CRISPR gene-column count."
    )


model_column = crispr_header[0]


if model_column not in [
    "Unnamed: 0",
    "ModelID"
]:

    raise RuntimeError(
        "Unexpected CRISPR model identifier column."
    )


new_columns = (
    newly_resolved["proposed_depmap_column"]
    .tolist()
)


if len(set(new_columns)) != 13:

    raise RuntimeError(
        "New gene mappings are not unique."
    )


missing_columns = (
    set(new_columns) - set(crispr_header)
)


if missing_columns:

    raise RuntimeError(
        "New gene columns missing from DepMap: "
        f"{sorted(missing_columns)}"
    )


print("\nNewly resolved gene columns:", len(new_columns))

print("CRISPR header correspondence: PASSED")


# ------------------------------------------------------------
# 6. LOAD ONLY THE NEW CRISPR MEASUREMENTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXTRACTING 13 NEW GENE-EFFECT COLUMNS")
print("-" * 65)


effects = pd.read_csv(
    CRISPR_PATH,
    usecols=[model_column] + new_columns,
    dtype={model_column: str},
    low_memory=False
)


effects = effects.rename(
    columns={
        model_column: "ModelID"
    }
)


effects["ModelID"] = (
    effects["ModelID"]
    .astype(str)
    .str.strip()
)


if effects["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate CRISPR model identifiers."
    )


effects = effects.set_index(
    "ModelID"
)


effects = effects.apply(
    pd.to_numeric,
    errors="raise"
)


if effects.shape != (1361, 13):

    raise RuntimeError(
        f"Unexpected extracted matrix: {effects.shape}"
    )


if np.isinf(
    effects.to_numpy(dtype=float)
).any():

    raise RuntimeError(
        "Infinite gene-effect values detected."
    )


print("\nExtracted matrix:", effects.shape)

print("Numerical values: VALIDATED")


# ------------------------------------------------------------
# 7. LOAD COLORECTAL AND DEDUPLICATED MODELS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RESTORING VALIDATED COLORECTAL COHORTS")
print("-" * 65)


crc_models = pd.read_csv(
    CRC_MODELS_PATH,
    dtype=str,
    keep_default_na=False
)


dedup_models = pd.read_csv(
    DEDUP_MODELS_PATH,
    dtype=str,
    keep_default_na=False
)


if (
    "ModelID" not in crc_models.columns
    or "ModelID" not in dedup_models.columns
):

    raise RuntimeError(
        "Model identifier column missing."
    )


crc_ids = crc_models["ModelID"].tolist()

dedup_ids = dedup_models["ModelID"].tolist()


if len(crc_ids) != 81:

    raise RuntimeError(
        "Expected 81 CRC adenocarcinoma models."
    )


if len(dedup_ids) != 79:

    raise RuntimeError(
        "Expected 79 patient-deduplicated models."
    )


if len(set(crc_ids)) != len(crc_ids):

    raise RuntimeError(
        "Duplicate CRC model IDs."
    )


if len(set(dedup_ids)) != len(dedup_ids):

    raise RuntimeError(
        "Duplicate sensitivity model IDs."
    )


if not set(dedup_ids).issubset(
    set(crc_ids)
):

    raise RuntimeError(
        "Deduplicated cohort is not a subset "
        "of the primary CRC cohort."
    )


if not set(crc_ids).issubset(
    set(effects.index)
):

    raise RuntimeError(
        "CRC models missing from CRISPR measurements."
    )


print("\nPrimary CRC models:", len(crc_ids))

print("Patient-deduplicated models:", len(dedup_ids))


# ------------------------------------------------------------
# 8. RECONSTRUCT THE SAME COMPARATOR AS PATCH 4.3
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RESTORING OTHER-LINEAGE COMPARATOR")
print("-" * 65)


models = pd.read_csv(
    MODEL_PATH,
    dtype=str,
    keep_default_na=False
)


if models["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate ModelIDs in metadata."
    )


models = models.set_index("ModelID")


if not set(effects.index).issubset(
    set(models.index)
):

    raise RuntimeError(
        "CRISPR models missing from metadata."
    )


screened = models.loc[
    effects.index
].copy()


crc_mask = (
    screened["OncotreePrimaryDisease"]
    .str.strip()
    .eq("Colorectal Adenocarcinoma")
)


bowel_mask = (
    screened["OncotreeLineage"]
    .str.contains(
        r"bowel",
        case=False,
        regex=True,
        na=False
    )
)


nonmalignant_mask = (
    screened["OncotreePrimaryDisease"]
    .str.contains(
        r"non[- ]?cancerous|"
        r"benign|adenoma|"
        r"normal tissue|"
        r"hyperplasia|"
        r"dysplasia|"
        r"sessile serrated",
        case=False,
        regex=True,
        na=False
    )
)


other_mask = (
    ~bowel_mask
    & ~nonmalignant_mask
    & ~crc_mask
)


other_ids = (
    screened.index[other_mask]
    .tolist()
)


if len(other_ids) != 1260:

    raise RuntimeError(
        "Other-lineage comparator differs "
        "from Patch 4.3."
    )


if set(crc_ids) != set(
    screened.index[crc_mask]
):

    raise RuntimeError(
        "Primary CRC model definition "
        "differs from Patch 4.3."
    )


print("\nOther-lineage models:", len(other_ids))

print("Comparator validation: PASSED")


# ------------------------------------------------------------
# 9. BUILD ANALYSIS MATRICES
# ------------------------------------------------------------

crc_effects = effects.loc[
    crc_ids
].copy()


dedup_effects = effects.loc[
    dedup_ids
].copy()


other_effects = effects.loc[
    other_ids
].copy()


print("\nNew-gene matrix dimensions:")

print("CRC:", crc_effects.shape)

print("Deduplicated CRC:", dedup_effects.shape)

print("Other lineages:", other_effects.shape)


# ------------------------------------------------------------
# 10. GENE-EFFECT SUMMARY FUNCTION
# ------------------------------------------------------------

def summarize_effect(values):

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[
        np.isfinite(values)
    ]

    n = len(values)

    if n == 0:

        return {
            "n": 0,
            "median": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "fraction_lt_05": np.nan,
            "fraction_lt_1": np.nan
        }

    return {

        "n": int(n),

        "median": float(
            np.median(values)
        ),

        "q25": float(
            np.percentile(values, 25)
        ),

        "q75": float(
            np.percentile(values, 75)
        ),

        "fraction_lt_05": float(
            np.mean(values < -0.5)
        ),

        "fraction_lt_1": float(
            np.mean(values < -1.0)
        )
    }


# ------------------------------------------------------------
# 11. CALCULATE NEW GENE DEPENDENCIES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING NEWLY RESOLVED GENE DEPENDENCIES")
print("-" * 65)


new_records = []


for _, row in newly_resolved.iterrows():

    column = row[
        "proposed_depmap_column"
    ]

    crc_stats = summarize_effect(
        crc_effects[column]
    )

    other_stats = summarize_effect(
        other_effects[column]
    )

    dedup_stats = summarize_effect(
        dedup_effects[column]
    )


    crc_med = crc_stats["median"]

    other_med = other_stats["median"]

    dedup_med = dedup_stats["median"]


    if (
        np.isfinite(crc_med)
        and np.isfinite(other_med)
    ):

        median_difference = (
            crc_med - other_med
        )

    else:

        median_difference = np.nan


    if (
        np.isfinite(crc_med)
        and np.isfinite(dedup_med)
    ):

        dedup_difference = (
            dedup_med - crc_med
        )

    else:

        dedup_difference = np.nan


    new_records.append({

        "gene_id":
            row["gene_id"],

        "ensembl_stable_id":
            row["ensembl_stable_id"],

        "gene_name":
            row["gene_name"],

        "hgnc_approved_symbol":
            row["hgnc_approved_symbol"],

        "depmap_current_symbol":
            row["proposed_depmap_symbol"],

        "depmap_column":
            column,

        "depmap_entrez_id":
            row["hgnc_entrez_id"],

        "crc_models_total":
            len(crc_ids),

        "crc_models_measured":
            crc_stats["n"],

        "crc_measurement_fraction":
            crc_stats["n"] / len(crc_ids),

        "crc_median_gene_effect":
            crc_med,

        "crc_q25_gene_effect":
            crc_stats["q25"],

        "crc_q75_gene_effect":
            crc_stats["q75"],

        "crc_fraction_below_minus_05":
            crc_stats["fraction_lt_05"],

        "crc_fraction_below_minus_1":
            crc_stats["fraction_lt_1"],

        "other_models_measured":
            other_stats["n"],

        "other_median_gene_effect":
            other_med,

        "other_fraction_below_minus_05":
            other_stats["fraction_lt_05"],

        "crc_minus_other_median_effect":
            median_difference,

        "deduplicated_models_measured":
            dedup_stats["n"],

        "deduplicated_median_gene_effect":
            dedup_med,

        "dedup_minus_primary_median":
            dedup_difference,

        "high_crc_coverage":
            crc_stats["n"] >= 65

    })


new_results = pd.DataFrame(
    new_records
)


if len(new_results) != 13:

    raise RuntimeError(
        "Expected 13 newly calculated gene records."
    )


print("\nNew gene dependency calculations: COMPLETE")


# ------------------------------------------------------------
# 12. DISPLAY NEWLY RESOLVED GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("NEW GENE FUNCTIONAL RESULTS")
print("-" * 65)


display_columns = [
    "gene_name",
    "depmap_current_symbol",
    "crc_models_measured",
    "crc_median_gene_effect",
    "crc_fraction_below_minus_05",
    "other_median_gene_effect",
    "deduplicated_median_gene_effect"
]


print("\n")

print(
    new_results[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# 13. LOAD PREVIOUS REFINED DEPENDENCY SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRESERVING PREVIOUS CRISPR RESULTS")
print("-" * 65)


previous = pd.read_csv(
    PREVIOUS_SUMMARY_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


if len(previous) != 948:

    raise RuntimeError(
        "Unexpected previous summary size."
    )


if previous["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate gene IDs in previous summary."
    )


previous = previous.set_index(
    "gene_id",
    drop=False
)


# Preserve the original summary.
# All modifications below occur in a copy.

final = previous.copy(deep=True)


# ------------------------------------------------------------
# 14. ATTACH VALIDATED IDENTITY METADATA
# ------------------------------------------------------------

crosswalk_indexed = crosswalk.set_index(
    "gene_id"
)


if set(final.index) != set(
    crosswalk_indexed.index
):

    raise RuntimeError(
        "Gene IDs differ between crosswalk "
        "and dependency summary."
    )


if not (
    final["ensembl_stable_id"]
    == crosswalk_indexed.loc[
        final.index,
        "ensembl_stable_id"
    ]
).all():

    raise RuntimeError(
        "Ensembl identifiers differ "
        "between research tables."
    )


final["previous_depmap_mapping_status"] = (
    final["depmap_mapping_status"].copy()
)


final["hgnc_identity_status"] = (
    crosswalk_indexed.loc[
        final.index,
        "identity_validation_status"
    ]
)


final["hgnc_approved_symbol"] = (
    crosswalk_indexed.loc[
        final.index,
        "hgnc_approved_symbol"
    ]
)


final["hgnc_entrez_id"] = (
    crosswalk_indexed.loc[
        final.index,
        "hgnc_entrez_id"
    ]
)


final["depmap_current_symbol"] = (
    crosswalk_indexed.loc[
        final.index,
        "proposed_depmap_symbol"
    ]
)


final["depmap_final_mapping_available"] = (
    final["hgnc_identity_status"].isin([
        "Verified existing mapping",
        "Newly resolved by NCBI Gene ID"
    ])
)


# ------------------------------------------------------------
# 15. INTEGRATE THE 13 NEW DEPENDENCY RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INTEGRATING NEWLY RESOLVED DEPENDENCIES")
print("-" * 65)


statistic_columns = [
    "crc_models_total",
    "crc_models_measured",
    "crc_measurement_fraction",
    "crc_median_gene_effect",
    "crc_q25_gene_effect",
    "crc_q75_gene_effect",
    "crc_fraction_below_minus_05",
    "crc_fraction_below_minus_1",
    "other_models_measured",
    "other_median_gene_effect",
    "other_fraction_below_minus_05",
    "crc_minus_other_median_effect",
    "deduplicated_models_measured",
    "deduplicated_median_gene_effect",
    "dedup_minus_primary_median",
    "high_crc_coverage"
]


missing_statistic_columns = (
    set(statistic_columns) - set(final.columns)
)


if missing_statistic_columns:

    raise RuntimeError(
        "Previous summary missing statistics: "
        f"{missing_statistic_columns}"
    )


for _, row in new_results.iterrows():

    gene_id = row["gene_id"]

    if gene_id not in final.index:

        raise RuntimeError(
            f"New gene missing from summary: {gene_id}"
        )

    # Insert the verified DepMap identifier.

    final.at[
        gene_id,
        "depmap_column"
    ] = row["depmap_column"]


    final.at[
        gene_id,
        "depmap_entrez_id"
    ] = row["depmap_entrez_id"]


    final.at[
        gene_id,
        "depmap_mapping_status"
    ] = "HGNC NCBI Gene ID validated"


    # Insert the newly calculated statistics.

    for column in statistic_columns:

        final.at[
            gene_id,
            column
        ] = row[column]


print("\nNewly resolved genes integrated:", len(new_results))


# ------------------------------------------------------------
# 16. VALIDATE ORIGINAL 930 STATISTICS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VERIFYING PREVIOUS RESULTS WERE PRESERVED")
print("-" * 65)


verified_ids = (
    verified["gene_id"]
    .tolist()
)


for column in statistic_columns:

    if column == "high_crc_coverage":

        if not (
            final.loc[
                verified_ids,
                column
            ].astype(bool).to_numpy()

            ==

            previous.loc[
                verified_ids,
                column
            ].astype(bool).to_numpy()
        ).all():

            raise RuntimeError(
                f"Existing coverage flag changed: {column}"
            )

    else:

        old_values = pd.to_numeric(
            previous.loc[
                verified_ids,
                column
            ],
            errors="raise"
        ).to_numpy(dtype=float)


        new_values = pd.to_numeric(
            final.loc[
                verified_ids,
                column
            ],
            errors="raise"
        ).to_numpy(dtype=float)


        if not np.allclose(
            old_values,
            new_values,
            equal_nan=True,
            rtol=0.0,
            atol=0.0
        ):

            raise RuntimeError(
                "Previously calculated statistics changed: "
                f"{column}"
            )


print("\nOriginal 930 gene statistics: UNCHANGED")


# ------------------------------------------------------------
# 17. FINAL GENE COVERAGE VALIDATION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("FINAL DEPMAP COVERAGE VALIDATION")
print("-" * 65)


if len(final) != 948:

    raise RuntimeError(
        "Final candidate count changed."
    )


mapping_available = int(
    final["depmap_final_mapping_available"].sum()
)


if mapping_available != 943:

    raise RuntimeError(
        "Expected 943 validated mappings."
    )


unmapped_final = final[
    ~final["depmap_final_mapping_available"]
].copy()


if len(unmapped_final) != 5:

    raise RuntimeError(
        "Expected five genes without "
        "DepMap gene-effect columns."
    )


# Genes without a DepMap column must not
# receive invented functional measurements.

if unmapped_final[
    "crc_models_measured"
].fillna(0).gt(0).any():

    raise RuntimeError(
        "Unmapped genes unexpectedly have "
        "CRISPR measurements."
    )


print("\nResearch candidate genes:", len(final))

print("Verified DepMap gene mappings:", mapping_available)

print("Without DepMap gene columns:", len(unmapped_final))


print(
    "Genes with CRC measurements:",
    int(
        final["crc_models_measured"]
        .fillna(0)
        .gt(0)
        .sum()
    )
)


print(
    "Genes meeting >=80% CRC coverage:",
    int(
        final["high_crc_coverage"]
        .fillna(False)
        .astype(bool)
        .sum()
    )
)


print("\nCandidate preservation: PASSED")

print("Gene identity validation: PASSED")


# ------------------------------------------------------------
# 18. REVIEW REMAINING UNMEASURED GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENES WITHOUT DEPMAP MEASUREMENTS")
print("-" * 65)


print("\n")

print(
    unmapped_final[
        [
            "gene_name",
            "hgnc_approved_symbol",
            "hgnc_entrez_id",
            "hgnc_identity_status"
        ]
    ].to_string(index=False)
)


print("\nIMPORTANT:")

print(
    "Absent DepMap gene-effect columns "
    "are not biological evidence of "
    "nonessentiality or lack of expression."
)


# ------------------------------------------------------------
# 19. PREPARE NEW-GENE CRC MATRIX
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING NEWLY RESOLVED CRC MATRIX")
print("-" * 65)


column_to_research_symbol = dict(
    zip(
        newly_resolved["proposed_depmap_column"],
        newly_resolved["gene_name"]
    )
)


crc_export = crc_effects.rename(
    columns=column_to_research_symbol
).copy()


crc_export.index.name = "ModelID"


if crc_export.shape != (81, 13):

    raise RuntimeError(
        "Unexpected new-gene CRC matrix."
    )


print("\nNewly resolved CRC matrix:", crc_export.shape)


# ------------------------------------------------------------
# 20. SAVE FINAL DEPMAP RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING FINAL PHASE 4 RESULTS")
print("-" * 65)


final_output = final.reset_index(
    drop=True
)


final_output.to_csv(

    RESULTS /
    "depmap_26q3_hgnc_integrated_dependency_summary.csv",

    index=False
)


new_results.to_csv(

    RESULTS /
    "depmap_26q3_hgnc_resolved_gene_dependency.csv",

    index=False
)


unmapped_final.reset_index(
    drop=True
).to_csv(

    RESULTS /
    "depmap_26q3_remaining_unmeasured_candidates.csv",

    index=False
)


crc_export.to_csv(

    RESULTS /
    "depmap_26q3_hgnc_resolved_crc_matrix.csv"
)


# Separate, compact final mapping reference.

mapping_columns = [
    "gene_id",
    "ensembl_stable_id",
    "gene_name",
    "hgnc_approved_symbol",
    "hgnc_entrez_id",
    "depmap_current_symbol",
    "depmap_column",
    "depmap_final_mapping_available",
    "hgnc_identity_status"
]


final_output[
    mapping_columns
].to_csv(

    RESULTS /
    "depmap_26q3_final_candidate_gene_mapping.csv",

    index=False
)


print("\nAll final Phase 4 tables saved.")


# ------------------------------------------------------------
# 21. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING FINAL DEPMAP PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.5",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "depmap_release": "Public 26Q3",

    "source_model_sha256":
        sha256_file(MODEL_PATH),

    "source_crispr_sha256":
        sha256_file(CRISPR_PATH),

    "official_source_checksum_verified": False,

    "candidate_genes": 948,

    "previously_verified_genes": 930,

    "newly_resolved_genes": 13,

    "final_verified_gene_mappings": 943,

    "genes_without_depmap_columns": 5,

    "refined_crc_adenocarcinoma_models": 81,

    "patient_deduplicated_crc_models": 79,

    "other_lineage_comparator_models": 1260,

    "new_dependency_statistics_calculated_for": 13,

    "existing_dependency_statistics_preserved": True,

    "gene_mapping_method": (
        "TCGA Ensembl gene ID -> HGNC approved "
        "record -> NCBI Gene ID -> DepMap column"
    ),

    "reference_gene_naming": (
        "Original TCGA gene symbols retained; "
        "current HGNC/DepMap symbols recorded "
        "in separate columns"
    ),

    "unmapped_gene_policy": (
        "Preserve candidates with unavailable "
        "CRISPR values; do not assign zero effects"
    ),

    "primary_crc_cohort": (
        "81 models annotated as "
        "Colorectal Adenocarcinoma"
    ),

    "sensitivity_crc_cohort": (
        "79 models after deterministic "
        "patient-level deduplication"
    ),

    "dependency_interpretation": (
        "Descriptive CRISPR fitness effects; "
        "not direct antibody efficacy measures"
    ),

    "msi_status_inferred": False,

    "therapeutic_target_ranking_performed": False,

    "original_phase4_results_overwritten": False,

    "limitations": [

        "The working CRISPR CSV was reconstructed "
        "from two upload fragments.",

        "Official source checksum comparison "
        "has not been performed.",

        "The colorectal model cohort is "
        "not confirmed to be entirely MSS.",

        "Models may not capture primary tumor "
        "heterogeneity or stromal biology.",

        "Other-lineage comparisons are descriptive.",

        "Gene-effect measurements do not "
        "directly measure antibody binding.",

        "Weak dependency does not rule out "
        "ADC antigen suitability.",

        "Five candidate genes do not have "
        "gene-effect columns in this release.",

        "No therapeutic targets have "
        "been validated."

    ]
}


provenance_path = (
    META /
    "depmap_26q3_final_integration_provenance.json"
)


with open(
    provenance_path,
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nFinal Phase 4 provenance saved.")


# ------------------------------------------------------------
# 22. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.5 SUMMARY")
print("=" * 65)


print("\nDepMap release: Public 26Q3")


print(
    "Research candidates preserved:",
    len(final_output)
)


print(
    "Verified DepMap gene mappings:",
    mapping_available
)


print(
    "Newly recovered gene measurements:",
    len(new_results)
)


print(
    "Genes without DepMap columns:",
    len(unmapped_final)
)


print(
    "\nCRC adenocarcinoma models:",
    len(crc_ids)
)


print(
    "Patient-deduplicated models:",
    len(dedup_ids)
)


print(
    "Other-lineage models:",
    len(other_ids)
)


print("\nSaved files:")

print(
    "1. depmap_26q3_hgnc_integrated_dependency_summary.csv"
)

print(
    "2. depmap_26q3_hgnc_resolved_gene_dependency.csv"
)

print(
    "3. depmap_26q3_remaining_unmeasured_candidates.csv"
)

print(
    "4. depmap_26q3_hgnc_resolved_crc_matrix.csv"
)

print(
    "5. depmap_26q3_final_candidate_gene_mapping.csv"
)

print(
    "6. depmap_26q3_final_integration_provenance.json"
)


print("\nExisting 930 gene statistics: VERIFIED UNCHANGED")

print("Additional 13 genes: INTEGRATED")

print("All 948 candidates: PRESERVED")

print("No missing CRISPR values imputed.")

print("No source files modified.")

print("No therapeutic target scores assigned.")


print("\nPATCH 4.5 COMPLETE")

print("=" * 65)